Append a batch of new click-stream events to raw.order_events.

In [6]:
import sys, random
from datetime import datetime, timedelta
import duckdb

In [7]:
n = 500
DB = "shopflow.duckdb"

In [8]:
con = duckdb.connect(DB)

In [9]:
con.execute("""
            create table if not exists raw.order_events(
                event_id bigint, order_id bigint, event_type varchar, event_at timestamp
            )
            """)

In [10]:
last_id, last_at = con.execute("""
                            select 
                                coalesce(max(event_id), 0),
                                coalesce(max(event_at), timestamp '2026-11-01 00:00:00')
                            from raw.order_events
                               """).fetchone()

In [ ]:
rows = []
for i in range(1, n + 1):
    rows.append((last_id + 1, random.randint(1, 1200),
                 random.choice(["viewed", "added_to_cart", "checked_out"]),
                 last_at + timedelta(seconds=i)
                 ))
    
con.executemany("insert into raw.order_events values (?,?,?,?)", rows)
total = con.execute("select count(*) from raw.order_events").fetchone()[0]

In [ ]:
print(f"added {n} events, raw.order_events new has {total} rows")

In [14]:
con.close()